# Intelligent Document Assistant using RAG
### End-to-end walkthrough: Ingestion → Chunking → Embeddings → FAISS → Retrieval → Generation

This notebook demonstrates the full RAG pipeline built for this project, phase by phase,
and includes a comparison of different chunk-size configurations as required by the
project guidelines.

## 0. Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv()

# Set your OpenAI API key here if not using a .env file
# os.environ["OPENAI_API_KEY"] = "sk-..."

from src.ingestion import load_documents
from src.chunking import chunk_documents, compare_chunk_sizes
from src.vector_store import build_vector_store, save_vector_store
from src.retrieval import retrieve_relevant_chunks
from src.generation import generate_answer, get_llm
from src.rag_pipeline import RAGPipeline

## Phase 1: Document Ingestion
Upload PDF documents, extract text, and clean/preprocess it.

In [ ]:
# Point this at your sample documents (e.g. an annual report or research paper PDF)
sample_files = ["../data/sample_document.pdf"]

documents = load_documents(sample_files)
for d in documents:
    print(f"{d.source}: {d.metadata['num_chars']} characters after cleaning")

## Phase 2: Document Chunking
Split documents into manageable, semantically coherent chunks.

In [ ]:
chunks = chunk_documents(documents, chunk_size=1000, chunk_overlap=150)
print(f"Created {len(chunks)} chunks")
print("\nSample chunk:\n", chunks[0].text[:400])

### Comparing multiple chunk sizes
Per the project guidelines, we compare a few chunk-size/overlap configurations
to justify the final choice used in the app.

In [ ]:
stats = compare_chunk_sizes(documents)
for config, s in stats.items():
    print(config, "->", s)

# Observation: smaller chunks (500 chars) give more, narrower chunks -- good
# for precise fact lookup but may lose surrounding context. Larger chunks
# (1500 chars) preserve more context per chunk but retrieval becomes less
# precise and prompts get longer/more expensive. 1000/150 is a reasonable
# middle ground and is what the app uses by default.

## Phase 3: Embedding Generation
Generate vector embeddings with an open-source sentence-transformers model
and store them in a FAISS vector database.

In [ ]:
vector_store = build_vector_store(chunks)
save_vector_store(vector_store, "../data/faiss_index")
print("FAISS index built and saved to ../data/faiss_index")

## Phase 4: Retrieval Pipeline
Given a natural-language question, retrieve and rank the most relevant chunks.

In [ ]:
query = "What is this document about?"
retrieved = retrieve_relevant_chunks(vector_store, query, top_k=4)
for r in retrieved:
    print(f"[{r.score}] {r.source} :: {r.text[:150]}...\n")

## Phase 5: Answer Generation
Pass the retrieved context to an LLM (OpenAI) using a grounded prompt template
so the answer is based on the retrieved content, not general knowledge.

In [ ]:
answer = generate_answer(query, retrieved)
print(answer)

## End-to-End Pipeline
The `RAGPipeline` class in `src/rag_pipeline.py` wraps all five phases above
into the object the Streamlit app (`app.py`) calls directly.

In [ ]:
pipeline = RAGPipeline(chunk_size=1000, chunk_overlap=150, top_k=4)
pipeline.ingest(sample_files)

result = pipeline.ask("Summarize the key points of this document.")
print("ANSWER:\n", result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(f"  - {s['source']} (score={s['score']})")

## Next Steps
- Run `streamlit run app.py` from the project root to use the interactive UI.
- See `README.md` for setup instructions and `EVALUATION_REPORT.md` for a
  summary of design decisions and results against the evaluation metrics.